# Phase 1 Final Controls Metric Contract Audit

Claim-closed orchestration notebook. This notebook calls the CLI-backed metric-contract audit for failed final controls. It does not recompute controls, change thresholds, edit logits or metrics, reinterpret failed controls as pass, or open Phase 1 claims.

Scientific integrity rule: a prospective config clarification alone does not reclassify old dedicated-control artifacts. Formula closure requires artifact-level formula metadata on the dedicated-control run under audit, or another explicitly non-prospective reviewed contract source.


In [ ]:
# Cell 1 - Bootstrap repo and run unit tests.

from pathlib import Path
import base64
import getpass
import hashlib
import json
import os
import subprocess
import sys
from datetime import datetime, timezone

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'
RUN_UNITTESTS = True

def run(cmd, cwd=None, check=True):
    display = []
    for item in cmd:
        text = str(item)
        if 'Authorization: Basic' in text:
            display.append('http.extraHeader=<redacted>')
        else:
            display.append(text)
    print('$ ' + ' '.join(display))
    result = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, check=check)
    return result

if not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private, hoac de trong neu public: ')
    if token:
        header = 'Authorization: Basic ' + base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
    else:
        run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
if RUN_UNITTESTS:
    unit_result = subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=str(REPO_DIR), text=True)
    if unit_result.returncode != 0:
        raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])


In [ ]:
# Cell 2 - Select reviewed source artifacts and keep launch disabled by default.

EXPECTED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = ARTIFACT_ROOT / 'prereg/20260418T161442014597Z/prereg_bundle.json'

EXPECTED_RELATIVE_METRIC_FORMULA_ID = 'raw_ba_ratio'
EXPECTED_RELATIVE_METRIC_FORMULA_DEFINITION = 'control_balanced_accuracy / baseline_balanced_accuracy'
REQUIRED_FORMULA_METADATA_CONTROLS = ['nuisance_shared_control', 'spatial_control']

# Preferred: pin reviewed sources explicitly. Leave as None to resolve the latest valid run.
# Do not type placeholder run-id strings into these fields.
PINNED_CONTROLS_REMEDIATION_AUDIT_RUN = None
PINNED_FINAL_DEDICATED_CONTROLS_RUN = None

def latest_run_with_file(root: Path, required_file: str) -> Path:
    candidates = sorted([
        p for p in root.iterdir()
        if p.is_dir() and (p / required_file).exists()
    ])
    if not candidates:
        raise FileNotFoundError(f'No runs with {required_file} under {root}')
    print(f'Found runs under {root}:', len(candidates))
    for p in candidates[-5:]:
        print(p, 'required file exists =', (p / required_file).exists())
    return candidates[-1]

if PINNED_CONTROLS_REMEDIATION_AUDIT_RUN is None:
    CONTROLS_REMEDIATION_AUDIT_RUN = latest_run_with_file(
        ARTIFACT_ROOT / 'phase1_final_controls_remediation_audit',
        'phase1_final_controls_remediation_audit_summary.json',
    )
else:
    CONTROLS_REMEDIATION_AUDIT_RUN = Path(PINNED_CONTROLS_REMEDIATION_AUDIT_RUN)

if PINNED_FINAL_DEDICATED_CONTROLS_RUN is None:
    FINAL_DEDICATED_CONTROLS_RUN = latest_run_with_file(
        ARTIFACT_ROOT / 'phase1_final_dedicated_controls',
        'final_dedicated_control_manifest.json',
    )
else:
    FINAL_DEDICATED_CONTROLS_RUN = Path(PINNED_FINAL_DEDICATED_CONTROLS_RUN)

OUTPUT_ROOT = ARTIFACT_ROOT / 'phase1_final_controls_metric_contract_audit'
PLAN_ROOT = ARTIFACT_ROOT / 'phase1_final_controls_metric_contract_audit_plan'

RUN_FINAL_CONTROLS_METRIC_CONTRACT_AUDIT = False
REQUIRED_ACK = 'I reviewed final controls metric contract audit and understand it diagnoses formula ambiguity without changing thresholds metrics logits or opening claims'
MANUAL_LAUNCH_ACK = ''
FIX_MARKER = 'phase1_final_controls_metric_contract_audit_v2_nonretro_formula_metadata_20260423'

print('Notebook fix marker:', FIX_MARKER)
print('Controls remediation audit run:', CONTROLS_REMEDIATION_AUDIT_RUN)
print('Controls remediation audit summary exists:', (CONTROLS_REMEDIATION_AUDIT_RUN / 'phase1_final_controls_remediation_audit_summary.json').exists())
print('Final dedicated controls run:', FINAL_DEDICATED_CONTROLS_RUN)
print('Final dedicated control manifest exists:', (FINAL_DEDICATED_CONTROLS_RUN / 'final_dedicated_control_manifest.json').exists())
print('Expected relative metric formula:', EXPECTED_RELATIVE_METRIC_FORMULA_ID)
print('Run flag:', RUN_FINAL_CONTROLS_METRIC_CONTRACT_AUDIT)


In [ ]:
# Cell 3 - Utilities and prereg/source validation.

def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

def write_json(path: Path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def resolve_prereg_bundle(path: Path) -> Path:
    if path.exists():
        return path
    candidates = sorted(ARTIFACT_ROOT.glob('prereg/*/prereg_bundle.json'))
    print('Found prereg bundles:', len(candidates))
    for candidate in candidates:
        try:
            payload = read_json(candidate)
        except Exception:
            continue
        if payload.get('prereg_bundle_hash_sha256') == EXPECTED_PREREG_IDENTITY_HASH:
            return candidate
    raise FileNotFoundError(f'No locked prereg bundle found for expected identity hash under {ARTIFACT_ROOT / "prereg"}')

def collect_dedicated_relative_formula_metadata(run_dir: Path) -> dict:
    metadata = {}
    for control_id, filename in {
        'nuisance_shared_control': 'nuisance_shared_control.json',
        'spatial_control': 'spatial_control.json',
    }.items():
        path = run_dir / filename
        if not path.exists():
            metadata[control_id] = {'missing_artifact': str(path)}
            continue
        threshold = read_json(path).get('threshold', {})
        metadata[control_id] = {
            'relative_metric_formula_id': threshold.get('relative_metric_formula_id'),
            'relative_metric_formula_definition': threshold.get('relative_metric_formula_definition'),
            'relative_metric_formula_source': threshold.get('relative_metric_formula_source'),
            'relative_to_baseline': threshold.get('relative_to_baseline'),
        }
    return metadata

PREREG_BUNDLE = resolve_prereg_bundle(Path(PREREG_BUNDLE))
CONTROLS_REMEDIATION_AUDIT_RUN = Path(CONTROLS_REMEDIATION_AUDIT_RUN)
FINAL_DEDICATED_CONTROLS_RUN = Path(FINAL_DEDICATED_CONTROLS_RUN)

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_sha256 = sha256(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

remediation_summary = read_json(CONTROLS_REMEDIATION_AUDIT_RUN / 'phase1_final_controls_remediation_audit_summary.json')
dedicated_summary = read_json(FINAL_DEDICATED_CONTROLS_RUN / 'phase1_final_dedicated_controls_summary.json')
dedicated_manifest = read_json(FINAL_DEDICATED_CONTROLS_RUN / 'final_dedicated_control_manifest.json')
source_dedicated_relative_formula_metadata = collect_dedicated_relative_formula_metadata(FINAL_DEDICATED_CONTROLS_RUN)
source_dedicated_missing_relative_formula_metadata = [
    control_id
    for control_id in REQUIRED_FORMULA_METADATA_CONTROLS
    if source_dedicated_relative_formula_metadata.get(control_id, {}).get('relative_metric_formula_id') != EXPECTED_RELATIVE_METRIC_FORMULA_ID
    or source_dedicated_relative_formula_metadata.get(control_id, {}).get('relative_metric_formula_definition') != EXPECTED_RELATIVE_METRIC_FORMULA_DEFINITION
]

assert remediation_summary.get('claims_opened') is False
assert remediation_summary.get('claim_ready') is False
assert remediation_summary.get('control_suite_passed') is False
assert remediation_summary.get('blocking_controls'), 'Controls remediation audit must record current blockers.'
assert dedicated_summary.get('claim_ready') is False
assert dedicated_manifest.get('claim_ready') is False
print('Controls remediation blockers:', remediation_summary.get('blocking_controls'))
print('Dedicated failed results:', dedicated_manifest.get('failed_results'))
print('Source dedicated relative formula metadata:')
print(json.dumps(source_dedicated_relative_formula_metadata, indent=2))
print('Source dedicated missing/stale relative formula metadata:', source_dedicated_missing_relative_formula_metadata)
if source_dedicated_missing_relative_formula_metadata:
    print('NOTE: Missing artifact-level formula metadata means any config-only formula clarification remains non-retroactive for this audit source.')

preflight = subprocess.run(
    ['python', '-m', 'src.cli', 'phase1_final_controls_metric_contract_audit', '--help'],
    cwd=str(REPO_DIR),
    text=True,
    capture_output=True,
)
runner_available = preflight.returncode == 0
print('Runner available:', runner_available)
assert runner_available, preflight.stderr


In [ ]:
# Cell 4 - Record manual-hold plan and exact CLI command.

created_utc = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
plan_dir = PLAN_ROOT / created_utc
cmd = [
    'python', '-m', 'src.cli', 'phase1_final_controls_metric_contract_audit',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--controls-remediation-audit-run', str(CONTROLS_REMEDIATION_AUDIT_RUN),
    '--final-dedicated-controls-run', str(FINAL_DEDICATED_CONTROLS_RUN),
    '--output-root', str(OUTPUT_ROOT),
    '--metric-contract-config', 'configs/phase1/final_controls_metric_contract_audit.json',
    '--dedicated-controls-config', 'configs/phase1/final_dedicated_controls.json',
    '--control-suite-config', 'configs/controls/control_suite_spec.yaml',
    '--gate2-config', 'configs/gate2/synthetic_validation.json',
]
plan = {
    'status': 'phase1_final_controls_metric_contract_audit_manual_hold_recorded',
    'created_utc': created_utc,
    'fix_marker': FIX_MARKER,
    'runner_available': runner_available,
    'run_flag': RUN_FINAL_CONTROLS_METRIC_CONTRACT_AUDIT,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'prereg_bundle': str(PREREG_BUNDLE),
    'locked_prereg_identity_hash': actual_prereg_hash,
    'raw_prereg_file_sha256': raw_prereg_sha256,
    'controls_remediation_audit_run': str(CONTROLS_REMEDIATION_AUDIT_RUN),
    'final_dedicated_controls_run': str(FINAL_DEDICATED_CONTROLS_RUN),
    'source_dedicated_relative_formula_metadata': source_dedicated_relative_formula_metadata,
    'source_dedicated_missing_relative_formula_metadata': source_dedicated_missing_relative_formula_metadata,
    'command': cmd,
    'scientific_limit': 'This plan authorizes only a claim-closed metric-contract audit; it does not authorize threshold changes or claims.',
    'non_retroactive_formula_rule': 'Prospective config-only formula clarification does not close ambiguity for older dedicated-control artifacts.',
}
write_json(plan_dir / 'phase1_final_controls_metric_contract_audit_manual_hold.json', plan)
print('Plan source of truth:', plan_dir)
print(json.dumps(plan, indent=2))


In [ ]:
# Cell 5 - Manual launch gate.

if not RUN_FINAL_CONTROLS_METRIC_CONTRACT_AUDIT:
    hold = {
        'status': 'phase1_final_controls_metric_contract_audit_manual_hold',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'run_flag': RUN_FINAL_CONTROLS_METRIC_CONTRACT_AUDIT,
        'required_ack': REQUIRED_ACK,
        'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
        'plan_dir': str(plan_dir),
        'claims_opened': False,
    }
    print(json.dumps(hold, indent=2))
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not launch metric-contract audit without explicit claim-closed acknowledgement.')
else:
    launch_review = {
        'status': 'phase1_final_controls_metric_contract_audit_launch_acknowledged',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'ack_matched': True,
        'claims_opened': False,
    }
    write_json(plan_dir / 'phase1_final_controls_metric_contract_audit_launch_review.json', launch_review)
    run(cmd, cwd=REPO_DIR)
    print('Metric-contract audit command completed. Review formula rows before any remediation decision.')


In [ ]:
# Cell 6 - Inspect latest metric-contract audit output.

latest_txt = OUTPUT_ROOT / 'latest.txt'
print('latest exists:', latest_txt.exists())
if latest_txt.exists():
    latest_run = Path(latest_txt.read_text(encoding='utf-8').strip())
else:
    runs = sorted([path for path in OUTPUT_ROOT.iterdir() if path.is_dir()]) if OUTPUT_ROOT.exists() else []
    latest_run = runs[-1] if runs else None

if latest_run is None:
    print('No metric-contract audit output yet; this is expected for manual hold.')
else:
    print('Latest final controls metric-contract audit output:', latest_run)
    expected_files = [
        'phase1_final_controls_metric_contract_audit_inputs.json',
        'phase1_final_controls_metric_contract_audit_summary.json',
        'phase1_final_controls_metric_contract_audit_report.md',
        'phase1_final_controls_metric_contract_source_links.json',
        'phase1_final_controls_metric_contract_input_validation.json',
        'relative_metric_formula_review.json',
        'controls_threshold_contract_review.json',
        'controls_metric_contract_remediation_recommendation.json',
        'phase1_final_controls_metric_contract_claim_state.json',
        'phase1_final_controls_metric_contract_decision_memo.md',
    ]
    for name in expected_files:
        print(name, 'exists =', (latest_run / name).exists())
    summary = read_json(latest_run / 'phase1_final_controls_metric_contract_audit_summary.json')
    formula_review = read_json(latest_run / 'relative_metric_formula_review.json')
    recommendation = read_json(latest_run / 'controls_metric_contract_remediation_recommendation.json')
    print(json.dumps({
        'status': summary.get('status'),
        'claim_ready': summary.get('claim_ready'),
        'claims_opened': summary.get('claims_opened'),
        'relative_formula_locked': summary.get('relative_formula_locked'),
        'locked_formula_id': formula_review.get('locked_formula_id'),
        'locked_formula_source': formula_review.get('locked_formula_source'),
        'prospective_formula_source': formula_review.get('prospective_formula_source'),
        'formula_ambiguity_detected': summary.get('formula_ambiguity_detected'),
        'controls_with_formula_dependent_pass_status': summary.get('controls_with_formula_dependent_pass_status'),
        'current_runtime_formula_ids': summary.get('current_runtime_formula_ids'),
        'next_step': summary.get('next_step'),
        'recommendation_blockers': recommendation.get('blockers'),
        'scientific_limit': summary.get('scientific_limit'),
    }, indent=2))
    print('Formula rows:')
    for row in formula_review.get('rows', []):
        print(json.dumps(row, indent=2))


In [ ]:
# Cell 7 - Assertions and local review note.

if RUN_FINAL_CONTROLS_METRIC_CONTRACT_AUDIT:
    assert latest_run is not None, 'Expected metric-contract audit output after launch.'
    summary = read_json(latest_run / 'phase1_final_controls_metric_contract_audit_summary.json')
    formula_review = read_json(latest_run / 'relative_metric_formula_review.json')
    threshold_review = read_json(latest_run / 'controls_threshold_contract_review.json')
    recommendation = read_json(latest_run / 'controls_metric_contract_remediation_recommendation.json')
    claim_state = read_json(latest_run / 'phase1_final_controls_metric_contract_claim_state.json')

    assert summary.get('status') == 'phase1_final_controls_metric_contract_audit_recorded', summary
    assert summary.get('claim_ready') is False
    assert summary.get('claims_opened') is False
    assert claim_state.get('headline_phase1_claim_open') is False
    assert recommendation.get('do_not_change_thresholds') is True
    assert recommendation.get('do_not_edit_logits_or_metrics') is True
    assert recommendation.get('do_not_reclassify_existing_controls') is True
    assert formula_review.get('rows'), 'Expected formula rows to be recorded.'
    assert threshold_review.get('all_runtime_thresholds_match_locked_config') in {True, False}
    if summary.get('relative_formula_locked') is True:
        assert formula_review.get('locked_formula_id') == EXPECTED_RELATIVE_METRIC_FORMULA_ID
        assert formula_review.get('locked_formula_source'), 'Locked formula requires a source.'
    else:
        if formula_review.get('prospective_formula_source'):
            assert not formula_review.get('locked_formula_source'), 'Prospective config-only source must not be treated as locked.'

    review_note = {
        'status': 'phase1_final_controls_metric_contract_audit_review_pass_claim_closed',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'reviewed_run': str(latest_run),
        'checks_passed': [
            'required_artifacts_present',
            'claim_ready_false',
            'claims_opened_false',
            'formula_rows_recorded',
            'threshold_contract_review_recorded',
            'recommendation_preserves_thresholds_logits_metrics',
            'claim_state_closed',
            'prospective_config_not_used_as_retroactive_artifact_lock',
        ],
        'relative_formula_locked': summary.get('relative_formula_locked'),
        'locked_formula_id': formula_review.get('locked_formula_id'),
        'locked_formula_source': formula_review.get('locked_formula_source'),
        'prospective_formula_source': formula_review.get('prospective_formula_source'),
        'source_dedicated_relative_formula_metadata': source_dedicated_relative_formula_metadata,
        'source_dedicated_missing_relative_formula_metadata': source_dedicated_missing_relative_formula_metadata,
        'formula_ambiguity_detected': summary.get('formula_ambiguity_detected'),
        'controls_with_formula_dependent_pass_status': summary.get('controls_with_formula_dependent_pass_status'),
        'current_runtime_formula_ids': summary.get('current_runtime_formula_ids'),
        'next_step': summary.get('next_step'),
        'allowed_interpretation': 'Metric-contract diagnostic only. It may identify ambiguity or code/config review targets.',
        'not_allowed_interpretation': 'Do not treat this audit as a control pass, threshold change, or Phase 1 efficacy evidence.',
        'not_ok_to_claim': claim_state.get('not_ok_to_claim', []),
    }
    write_json(latest_run / 'phase1_final_controls_metric_contract_audit_review_note.json', review_note)
    print('Review note written:', latest_run / 'phase1_final_controls_metric_contract_audit_review_note.json')
    print(json.dumps(review_note, indent=2))
else:
    print('Assertions skipped because launch flag is False. This is expected during first-pass manual hold.')


In [ ]:
# Cell 8 - Closeout message.

print('================ Phase 1 Final Controls Metric Contract Audit Closeout ================')
print('Notebook fix marker:', FIX_MARKER)
print('Plan source of truth:', plan_dir)
print('Runner available:', runner_available)
print('Run flag:', RUN_FINAL_CONTROLS_METRIC_CONTRACT_AUDIT)
print('Expected locked prereg identity hash:', EXPECTED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
print('Controls remediation audit run:', CONTROLS_REMEDIATION_AUDIT_RUN)
print('Final dedicated controls run:', FINAL_DEDICATED_CONTROLS_RUN)
print('Source dedicated formula metadata complete:', not source_dedicated_missing_relative_formula_metadata)
print('Source dedicated missing/stale formula metadata:', source_dedicated_missing_relative_formula_metadata)

if not RUN_FINAL_CONTROLS_METRIC_CONTRACT_AUDIT:
    print('HELD: Metric-contract audit was not launched because manual flag is False.')
    print('NEXT: review the plan, then rerun only with explicit claim-closed acknowledgement if appropriate.')
else:
    print('Latest metric-contract audit output:', latest_run)
    print('Relative formula locked:', summary.get('relative_formula_locked'))
    print('Locked formula id:', formula_review.get('locked_formula_id'))
    print('Locked formula source:', formula_review.get('locked_formula_source'))
    print('Prospective formula source:', formula_review.get('prospective_formula_source'))
    print('Formula ambiguity detected:', summary.get('formula_ambiguity_detected'))
    print('Controls with formula-dependent pass status:', summary.get('controls_with_formula_dependent_pass_status'))
    print('Current runtime formula ids:', summary.get('current_runtime_formula_ids'))
    print('Next step:', summary.get('next_step'))
    print('CHECK REQUIRED: Review phase1_final_controls_metric_contract_decision_memo.md before any code/config remediation.')
print('NOT OK TO CLAIM: This notebook does not prove decoder efficacy, A2d efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')
